# Part 6 — Cross-Species Workflows: Humanization

*Last updated:* 2026-11-12

This tutorial shows a practical **humanization** workflow: mapping mouse/pig genes into a human gene space so you can run
human-centric downstream analyses (pathways, marker lists, integration, annotation).

**Learning objectives**
- Understand what humanization is (and when it is appropriate).
- Run a step-by-step mouse → human and pig → human mapping.
- Validate results and handle 1→n orthology ambiguity explicitly.
- Prepare outputs in a tidy, analysis-friendly format for comparative workflows.

> **Warning:** Orthology is not always one-to-one. This notebook focuses on making ambiguity visible and manageable.


## 6.1 — Humanization workflow (what it is, and when to use it)

This notebook shows a **reproducible, step-by-step pipeline**:

1. **Within-species cleanup (IDTrack)**
   - Map mouse/pig identifiers to consistent Ensembl gene IDs within that species, at a pinned snapshot release.

2. **Cross-species mapping (orthologs, from an orthology resource)**
   - Map mouse/pig Ensembl gene IDs to human ortholog Ensembl gene IDs with an orthology table of the same Ensembl release
     (Ensembl Compara's homologies, exported from BioMart or downloaded from Ensembl's FTP site). IDTrack does not compute
     orthology; it makes sure that the ids on both sides of the table are the ids of the release the table was made for.

3. **Human naming (optional, IDTrack)**
   - Convert human Ensembl gene IDs into HGNC symbols (or other human namespaces).

This notebook does **not** decide the ‘correct’ ortholog for you in complex families — it shows how to keep the candidates
visible and record the rule you apply.

### 6.1.1 Pre-requisites

### 6.1.1.1 IDTrack graphs
You should have built graphs for the organisms you use:
- `mus_musculus` and/or `sus_scrofa`
- `homo_sapiens`

### 6.1.1.2 An orthology table
Ensembl publishes the orthologs of every release (Ensembl Compara): in BioMart (the attributes "Human gene stable ID" and
"Human homology type" of a mouse or pig gene dataset) and as files under
`https://ftp.ensembl.org/pub/release-<release>/tsv/ensembl-compara/homologies/`. Use the table of the release of your graph's
snapshot, so that its gene ids are the ids IDTrack converts to.

In [1]:
# The orthology table of this tutorial: the two rows of Ensembl Compara's homologies of release 115 for human TP53 (mouse
# Trp53 and pig TP53, both one-to-one orthologs), with the columns of a BioMart export (gene stable ID, human gene stable
# ID, human homology type). In your work, read the export of your snapshot release, e.g. pd.read_csv('mouse_human_orthologs_115.tsv', sep='\t').
from __future__ import annotations

import pandas as pd

orthology = pd.DataFrame(
    [
        {'source_gene': 'ENSMUSG00000059552', 'human_gene': 'ENSG00000141510', 'homology_type': 'ortholog_one2one'},
        {'source_gene': 'ENSSSCG00000017950', 'human_gene': 'ENSG00000141510', 'homology_type': 'ortholog_one2one'},
    ]
)
orthology

,source_gene,human_gene,homology_type
0,ENSMUSG00000059552,ENSG00000141510,ortholog_one2one
1,ENSSSCG00000017950,ENSG00000141510,ortholog_one2one


### 6.1.2 Step-by-step: mouse → human

### 6.1.2.1 Convert a mouse identifier to a mouse Ensembl gene ID (IDTrack)

Start with whatever you have (often an MGI symbol). Convert to a **base Ensembl gene ID** in mouse.


In [2]:
import os
from pathlib import Path

import idtrack
from idtrack import API

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

API.configure_logger()

MOUSE_SNAPSHOT = 115  # pinned: the release of the orthology table
api_mouse = API.build('mus_musculus', MOUSE_SNAPSHOT, str(LOCAL_REPOSITORY))

mouse_query = 'Trp53'  # example MGI symbol; replace with your gene
mouse_to_ensembl = api_mouse.convert(mouse_query, to_release=MOUSE_SNAPSHOT, final_database='base_ensembl_gene')
print(mouse_to_ensembl.status, list(mouse_to_ensembl.targets()))

converted ['ENSMUSG00000059552']


`targets()` holds your mouse Ensembl gene ID (without version, because of `final_database='base_ensembl_gene'`).
If you get several, you are in a 1→n case (`tie=True` when they are equal in all evidence) — you may need to decide how to handle it;
the first entry is the first of the best answer in a fixed order (by gene id), not a choice by evidence.

### 6.1.2.2 Find human ortholog(s) (the orthology table)

Look the mouse gene up in the orthology table of the same release. Keep every ortholog the table lists, with its type.

In [3]:
mouse_targets = mouse_to_ensembl.targets()
mouse_ensembl_gene_id = mouse_targets[0] if len(mouse_targets) == 1 else None

if mouse_ensembl_gene_id is None:
    print('No single mouse Ensembl gene ID found; check your input ID and the mouse graph.')
    human_ensembl_gene_ids = []
else:
    rows = orthology[orthology['source_gene'] == mouse_ensembl_gene_id]
    print(rows.to_string(index=False))
    human_ensembl_gene_ids = sorted(rows['human_gene'])

human_ensembl_gene_ids

       source_gene      human_gene    homology_type
ENSMUSG00000059552 ENSG00000141510 ortholog_one2one


['ENSG00000141510']

### 6.1.2.3 Convert human Ensembl IDs into HGNC symbols (IDTrack)

Now we switch to the human graph and convert the human Ensembl IDs into HGNC symbols (optional but common).


In [4]:
if not human_ensembl_gene_ids:
    print('No human ortholog IDs available; skipping human conversion step.')
else:
    HUMAN_SNAPSHOT = 115  # pinned
    api_human = API.build('homo_sapiens', HUMAN_SNAPSHOT, str(LOCAL_REPOSITORY))

    # Convert all candidate orthologs (if many-to-many, you will see it here)
    human_results = api_human.convert_many(
        human_ensembl_gene_ids,
        to_release=HUMAN_SNAPSHOT,
        final_database='HGNC Symbol',
    )
    for r in human_results:
        print(r.query_id, '->', list(r.targets()), '| tie:', r.tie)

ENSG00000141510 -> ['TP53'] | tie: False


### 6.1.3 Step-by-step: pig → human (same pattern)

Use a pig API and start from your pig identifiers.
Many pig pipelines start from Ensembl IDs or Entrez IDs; adjust `final_database` accordingly.

In [5]:
PIG_SNAPSHOT = 115  # pinned
api_pig = API.build('sus_scrofa', PIG_SNAPSHOT, str(LOCAL_REPOSITORY))

pig_query = 'TP53'  # example; replace with your pig gene symbol or ID
pig_to_ensembl = api_pig.convert(pig_query, to_release=PIG_SNAPSHOT, final_database='base_ensembl_gene')
print(pig_to_ensembl.status, list(pig_to_ensembl.targets()))
print(orthology[orthology['source_gene'].isin(pig_to_ensembl.targets())].to_string(index=False))

converted ['ENSSSCG00000017950']
       source_gene      human_gene    homology_type
ENSSSCG00000017950 ENSG00000141510 ortholog_one2one


From here, reuse the same orthology lookup + human conversion steps as in the mouse section.

### 6.1.4 Advanced: choose among multiple ortholog candidates

When the table lists several orthologs (`ortholog_one2many`, `ortholog_many2many`), choose with evidence the orthology resource
provides (Ensembl Compara reports the percent identity of each pair, the gene-order conservation and whether the ortholog is
high-confidence) and record the rule you used; IDTrack keeps every candidate visible but does not rank orthologs.

In [6]:
# Example: keep every ortholog, and the rule that would pick one, as columns of the result
candidates = orthology.groupby('source_gene')['human_gene'].apply(list)
candidates.rename('human_candidates').to_frame().assign(rule='one2one only: keep; otherwise review')

,human_candidates,rule
source_gene,,
ENSMUSG00000059552,[ENSG00000141510],one2one only: keep; otherwise review
ENSSSCG00000017950,[ENSG00000141510],one2one only: keep; otherwise review


### 6.1.5 Practical cautions (please read)

1. **Orthology is context-dependent**: paralogs, gene family expansions, and annotation differences matter.
2. **Do not silently pick one** in many-to-many cases without recording the rule you used.
3. **Record provenance**: snapshot releases + YAML configs + the orthology resource and its release.

## 6.2 — Comparative Analysis Preparation

Once you have humanized identifiers, you can prepare a comparative workflow that is **reproducible** and **auditable**.

Recommended preparation steps:

1. **Within each species:** harmonize identifiers into stable Ensembl gene IDs at a fixed snapshot boundary.
2. **Across species:** map to human orthologs, but **keep ambiguity visible** (store 1→n mappings as lists).
3. **Record provenance:** snapshot boundaries, assemblies, and the orthology source/method.
4. **Define a policy for ambiguous cases:** drop / keep-all / choose-best (and justify it).

A practical output format is a tidy table with one row per input gene and explicit columns for provenance.
The next cell shows a minimal schema you can reuse.


In [7]:
# Example: a tidy, audit-friendly mapping table schema

import pandas as pd

mapping_table = pd.DataFrame(
    [
        {
            'source_species': 'mus_musculus',
            'source_namespace': 'MGI Symbol',
            'source_id': 'Trp53',
            'human_ensembl_gene_id': 'ENSG00000141510',
            'human_hgnc_symbol': 'TP53',
            'orthology_candidates': ['ENSG00000141510'],
            'snapshot_release_human': None,
            'snapshot_release_source': None,
            'notes': 'Example row; fill with your real results.'
        },
        {
            'source_species': 'sus_scrofa',
            'source_namespace': 'Ensembl Gene ID',
            'source_id': 'ENSSSCG00000000001',
            'human_ensembl_gene_id': None,
            'human_hgnc_symbol': None,
            'orthology_candidates': [],
            'snapshot_release_human': None,
            'snapshot_release_source': None,
            'notes': 'Example 1→0 / not found; keep these rows for reporting.'
        },
    ]
)

mapping_table


,source_species,source_namespace,source_id,human_ensembl_gene_id,human_hgnc_symbol,orthology_candidates,snapshot_release_human,snapshot_release_source,notes
0,mus_musculus,MGI Symbol,Trp53,ENSG00000141510,TP53,[ENSG00000141510],None,None,Example row; fill with your real results.
1,sus_scrofa,Ensembl Gene ID,ENSSSCG00000000001,NaN,NaN,[],None,None,Example 1→0 / not found; keep these rows for r...
